# EEG Preprocessing and Feature Exptraction

Preprocessing the N-back EEG data ,extracting bandpower features from different scalp regions and building the Feature-Matrix.

In [6]:
import mne
import numpy as np
from pathlib import Path
from scipy.signal import welch

In [ ]:
# Load example recording
zero_path = Path("..") / "data" / "sub-01" / "ses-S1" / "eeg" / "zeroBACK.set"
raw_zero = mne.io.read_raw_eeglab(zero_path, preload=True)

# Extract events
events_zero, event_id_zero = mne.events_from_annotations(raw_zero)

# Keep only EEG channels and filter
raw_zero_eeg = raw_zero.copy().pick("eeg")
raw_zero_eeg.filter(l_freq=1.0, h_freq=40.0)

# Create epochs from normal trials
zero_normal_trial_event = event_id_zero["6021"]

zero_epochs = mne.Epochs(
    raw_zero_eeg,
    events_zero,
    event_id=zero_normal_trial_event,
    tmin=0,
    tmax=2.0,
    baseline=None,
    preload=True
)

In [8]:
# PSD

zero_data = zero_epochs.get_data()

freqs, psd_zero = welch(
    zero_data,
    fs=raw_zero_eeg.info["sfreq"],
    nperseg=500,
    axis=-1
)

In [9]:
# Group channels based on Scalp Regions
rois = {
    "frontal": [
        "Fp1", "Fp2",
        "AF7", "AF3", "AFz", "AF4", "AF8",
        "F7", "F5", "F3", "F1", "Fz", "F2", "F4", "F6", "F8"
    ],

    "central": [
        "FC5", "FC3", "FC1", "FCz", "FC2", "FC4", "FC6",
        "C5", "C3", "C1", "C2", "C4", "C6"
    ],

    "temporal": [
        "FT9", "FT7", "FT8", "FT10",
        "T7", "T8",
        "TP7", "TP8", "TP10"
    ],

    "parietal": [
        "CP5", "CP3", "CP1", "CPz", "CP2", "CP4", "CP6",
        "P7", "P5", "P3", "P1", "Pz", "P2", "P4", "P6", "P8"
    ],

    "occipital": [
        "PO7", "PO3", "POz", "PO4", "PO8",
        "O1", "Oz", "O2"
    ]
}

# Powerbands
bands = {
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30)
}

In [ ]:
# Building the Feature Matrix

features_zero = []

for epoch_idx in range(len(zero_epochs)):
    epoch_feature = []
    # Map ROI channels
    for roi_name, channels in rois.items():
        channel_indices = [
            zero_epochs.ch_names.index(ch)
            for ch in channels
        ]
        roi_psd = np.mean(
            psd_zero[epoch_idx, channel_indices, :],
            axis=0
        )
        # Frequenzy Band
        for band_name, (fmin, fmax) in bands.items():
            freq_mask = (freqs >= fmin) & (freqs < fmax)
            bandpower = np.trapezoid(
                roi_psd[freq_mask],
                freqs[freq_mask]
            )
            epoch_feature.append(bandpower)

    # One epoch = 5 ROIs × 3 frequency bands        
    features_zero.append(epoch_feature)

X_zero = np.array(features_zero)    

In [12]:
# Create feature names and inspect the feature matrix
feature_names = [
    f"{roi}_{band}"
    for roi in rois
    for band in bands
]

print(X_zero.shape)
print(feature_names)

(96, 15)
['frontal_theta', 'frontal_alpha', 'frontal_beta', 'central_theta', 'central_alpha', 'central_beta', 'temporal_theta', 'temporal_alpha', 'temporal_beta', 'parietal_theta', 'parietal_alpha', 'parietal_beta', 'occipital_theta', 'occipital_alpha', 'occipital_beta']
